# ViDDAR dataset demo

`viddar_live_demo.ipynb` uses the **COCO** method from comparison v3: several synonym prompts, SAM picks the box it trusts most, and a painted card stands in for AR content.

This notebook uses the method that was actually run on the **ViDDAR** dataset:

1. The query is the dataset object name (`fire extinguisher`, `exit sign`, …). One prompt, thresholds **0.35 / 0.25**, highest DINO score.
2. Zero-shot SAM ViT-B turns that single box into one mask. No synonym list and no SAM box ranking.
3. Obstruction comes from the **real AR photo**, not a painted rectangle. Pixels that differ from the raw photo by more than 30 are the AR content. It obstructs when that region covers at least **25%** of the mask.

The five scenes in `viddar_images/` are pairs from the ViDDAR obstruction set (classes COCO does not have). Each folder has the raw photo and `AR_<name>.png`.

**Before you record:** Runtime → GPU. Run **[1]** once, then **[2]**, then change `FILE_NAME` in **[3]**.


## [1] Setup

In [ ]:
import io, os, sys, subprocess
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image

print("CUDA:", torch.cuda.is_available(), "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch the runtime to a GPU")
assert torch.cuda.is_available(), "This demo needs a GPU. Runtime → Change runtime type → T4."

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None
    IN_COLAB = False

subprocess.run(["pip", "install", "-q", "transformers"], check=False)

if not os.path.exists("/content/segment-anything"):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/facebookresearch/segment-anything.git",
                    "/content/segment-anything"], check=True)

SAM_CKPT = "/content/sam_vit_b_01ec64.pth"
if not os.path.exists(SAM_CKPT):
    print("Downloading SAM ViT-B (~375 MB)...")
    subprocess.run(["wget", "-q",
                    "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
                    "-O", SAM_CKPT], check=True)

sys.path.insert(0, "/content/segment-anything")
from segment_anything import sam_model_registry
from segment_anything.utils.transforms import ResizeLongestSide
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

print("Loading Grounding DINO...")
dino_processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    "IDEA-Research/grounding-dino-tiny").to("cuda").eval()

print("Loading SAM...")
sam = sam_model_registry["vit_b"](checkpoint=SAM_CKPT).cuda().eval()
resize_transform = ResizeLongestSide(sam.image_encoder.img_size)

# ViDDAR replication settings (not the COCO v3 settings)
BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25
DIFF_THRESHOLD = 30
ALPHA = 0.25

print("Ready. Run section [2] to load the ViDDAR pairs.")


## [2] Load the five ViDDAR scenes

Locally this finds `Demo/viddar_images` next to the notebook. In Colab, upload `labels.csv` and every png from `raw/` and `ar/` when the folder is not already there.


In [ ]:
def _find_root():
    candidates = [
        Path("viddar_images"),
        Path("/content/viddar_images"),
        Path("/content/Demo/viddar_images"),
    ]
    here = Path.cwd()
    candidates += [here / "viddar_images", here / "Demo" / "viddar_images"]
    for p in candidates:
        if (p / "labels.csv").exists() and (p / "raw").is_dir():
            return p.resolve()
    return None


ROOT = _find_root()
if ROOT is None and IN_COLAB:
    print("Select labels.csv and the png files from Demo/viddar_images/raw and Demo/viddar_images/ar")
    uploaded = colab_files.upload()
    ROOT = Path("/content/viddar_images")
    (ROOT / "raw").mkdir(parents=True, exist_ok=True)
    (ROOT / "ar").mkdir(parents=True, exist_ok=True)
    for name, blob in uploaded.items():
        if name == "labels.csv":
            (ROOT / name).write_bytes(blob)
        elif name.startswith("AR_"):
            (ROOT / "ar" / name).write_bytes(blob)
        else:
            (ROOT / "raw" / name).write_bytes(blob)

assert ROOT is not None and (ROOT / "labels.csv").exists(), (
    "Could not find viddar_images. Put the notebook next to that folder, or upload its files.")

labels = pd.read_csv(ROOT / "labels.csv")
labels.columns = [c.strip() for c in labels.columns]
print(f"Loaded {len(labels)} ViDDAR scenes from {ROOT}")
print(labels.to_string(index=False))


## [3] Run one scene

Set `FILE_NAME` to a row from the table above, then run the cell. The title compares the predicted obstruction with the dataset label.


In [ ]:
FILE_NAME = "fire-extinguisher_016.png"

import matplotlib.pyplot as plt


def load_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def dino_box(image_pil, query):
    """ViDDAR setting: one prompt, keep the highest score."""
    inputs = dino_processor(images=image_pil, text=[[query.lower()]], return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = dino_model(**inputs)
    results = dino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
        target_sizes=[image_pil.size[::-1]])[0]
    if len(results["boxes"]) == 0:
        return None, 0.0
    idx = int(results["scores"].argmax())
    return results["boxes"][idx].cpu().numpy(), float(results["scores"][idx])


def embed_image(image_rgb):
    inp = resize_transform.apply_image(image_rgb)
    h, w = inp.shape[:2]
    x = torch.as_tensor(inp, device="cuda").permute(2, 0, 1).contiguous()[None]
    with torch.no_grad():
        emb = sam.image_encoder(sam.preprocess(x))
    return {"emb": emb, "input_size": (h, w), "original_size": tuple(image_rgb.shape[:2])}


def sam_mask(sample, box):
    """One box, one mask. This is the ViDDAR call, not the COCO multimask ranking."""
    box_t = resize_transform.apply_boxes(np.asarray(box, dtype=np.float32)[None, :], sample["original_size"])
    box_t = torch.as_tensor(box_t, dtype=torch.float, device="cuda")
    with torch.no_grad():
        sparse, dense = sam.prompt_encoder(points=None, boxes=box_t, masks=None)
        low_res, iou_pred = sam.mask_decoder(
            image_embeddings=sample["emb"],
            image_pe=sam.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse,
            dense_prompt_embeddings=dense,
            multimask_output=False,
        )
        up = sam.postprocess_masks(low_res, input_size=sample["input_size"],
                                   original_size=sample["original_size"])
    mask = (up[0, 0].cpu().numpy() > 0).astype(np.uint8)
    score = float(iou_pred[0, 0].detach().cpu())
    return mask, score


def overlay_mask(image, mask, color, alpha=0.45):
    out = image.copy()
    painted = out.copy()
    painted[mask.astype(bool)] = color
    out = cv2.addWeighted(painted, alpha, out, 1 - alpha, 0)
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, color, 2)
    return out


def draw_box(image, box, color=(14, 165, 233), width=4):
    out = image.copy()
    x1, y1, x2, y2 = [int(round(v)) for v in box]
    cv2.rectangle(out, (x1, y1), (x2, y2), color, width)
    return out


def ar_content_from_pair(raw_rgb, ar_rgb, diff_threshold=DIFF_THRESHOLD):
    if ar_rgb.shape[:2] != raw_rgb.shape[:2]:
        ar_rgb = cv2.resize(ar_rgb, (raw_rgb.shape[1], raw_rgb.shape[0]), interpolation=cv2.INTER_LINEAR)
    gray = cv2.cvtColor(cv2.absdiff(raw_rgb, ar_rgb), cv2.COLOR_RGB2GRAY)
    return (gray > diff_threshold).astype(np.uint8), ar_rgb


row = labels[labels["File Name"] == FILE_NAME]
assert len(row) == 1, f"{FILE_NAME} is not in {ROOT / 'labels.csv'}"
row = row.iloc[0]
query = str(row["Object Name"])
dataset_label = str(row["obstruction"]).strip().lower()

raw = load_rgb(ROOT / "raw" / FILE_NAME)
ar = load_rgb(ROOT / "ar" / f"AR_{FILE_NAME}")
content, ar = ar_content_from_pair(raw, ar)

box, dino_score = dino_box(Image.fromarray(raw), query)

fig, axes = plt.subplots(1, 4, figsize=(16.5, 4.8))
for ax in axes:
    ax.axis("off")

if box is None:
    axes[0].imshow(raw); axes[0].set_title("1  Raw", fontsize=13)
    axes[1].imshow(raw); axes[1].set_title(f'No "{query}" at 0.35', fontsize=13, color="#b91c1c")
    axes[2].imshow(ar); axes[2].set_title("3  Real AR photo", fontsize=13)
    axes[3].imshow(overlay_mask(raw, content, (37, 99, 235), 0.45))
    axes[3].set_title("4  AR pixels (diff > 30)", fontsize=13)
    fig.suptitle(f'{FILE_NAME}  ·  dataset label: {dataset_label}  ·  no box, so no obstruction call',
                 fontsize=14, fontweight="bold", color="#b91c1c", y=1.04)
    print(f'{FILE_NAME}  query="{query}"  DINO found nothing  dataset={dataset_label}')
else:
    mask, sam_score = sam_mask(embed_image(raw), box)
    key = mask.astype(bool)
    cover = float((key & content.astype(bool)).sum() / (key.sum() + 1e-6)) if key.any() else 0.0
    predicted = "yes" if cover >= ALPHA else "no"
    agrees = predicted == dataset_label

    boxed = draw_box(raw, box)
    x1, y1 = int(box[0]), int(box[1])
    cv2.putText(boxed, f"{query}  {dino_score:.2f}",
                (max(4, x1), max(22, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (14, 165, 233), 2, cv2.LINE_AA)
    both = overlay_mask(overlay_mask(raw, content, (37, 99, 235), 0.40), mask, (249, 115, 22), 0.40)

    panels = [
        (raw, "1  Raw"),
        (overlay_mask(boxed, mask, (249, 115, 22), 0.45), "2  DINO box + SAM"),
        (ar, "3  Real AR photo"),
        (both, "4  Mask vs AR pixels"),
    ]
    for ax, (img, title) in zip(axes, panels):
        ax.imshow(img)
        ax.set_title(title, fontsize=13, pad=8)

    color = "#15803d" if agrees else "#b91c1c"
    verdict = "OBSTRUCTED" if predicted == "yes" else "CLEAR"
    match = "matches the dataset" if agrees else "differs from the dataset"
    fig.suptitle(
        f'Find “{query}”     {verdict} ({cover:.0%})     label {dataset_label}     {match}',
        fontsize=14, fontweight="bold", color=color, y=1.05,
    )
    print(
        f'{FILE_NAME}  query="{query}"  DINO={dino_score:.2f}  SAM score={sam_score:.2f}  '
        f'overlap={cover:.1%}  predicted={predicted}  dataset={dataset_label}  {match}'
    )

plt.tight_layout()
plt.show()
